# Notebook 05: Trade Evaluation

**Phase 2** — Trade Mechanics & Package Builder

This notebook documents the Lakers' 2026 offseason trade strategy:
- What assets do they have?
- What CBA rules constrain their moves?
- How valuable are their draft picks?
- What packages can they offer for each target archetype?

> All CBA rules and roster/pick information verified from ESPN Bobby Marks offseason guide + Sports Business Classroom, May 2026.


## A. Load Data and Modules

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.WARNING)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils.constants import (
    LAKERS_TRADEABLE, LAKERS_DRAFT_PICKS, LAKERS_EXTENSION_ELIGIBLE,
    RESTRICTED_FREE_AGENTS, SALARY_CAP_2026_27, FIRST_APRON_2026_27, SECOND_APRON_2026_27,
)
from src.utils.cba_rules import (
    trade_is_legal, salary_match_options, determine_apron_status,
    available_exceptions, lakers_cap_scenarios,
)
from src.models.pick_value import (
    PICK_VALUE_CURVE, estimate_pick_value, lakers_tradeable_picks_summary,
    pick_combination_value, check_stepien_rule,
)
from src.models.trade_evaluator import (
    build_trade_packages, evaluate_single_trade, packages_to_dataframe,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

# Load the Phase 1 valued dataset
df = pd.read_parquet('../data/cache/player_valued_2025-26.parquet')
print(f"Loaded {len(df)} players.")
print(f"Lakers players: {(df['TEAM_ABBREVIATION'] == 'LAL').sum()}")
print(f"Players with 2026-27 salary: {df['SALARY_FORWARD'].notna().sum()}")
print(f"Players with archetype classification: {df['BEST_ARCHETYPE'].notna().sum()}")


Loaded 582 players.
Lakers players: 18
Players with 2026-27 salary: 325
Players with archetype classification: 226


## B. Lakers Asset Inventory and Cap Situation

The Lakers' flexibility depends on free agent decisions not yet made. We model two paths:
- **Cap space path**: Renounce most FAs, operate as cap-space team (~$47-67M room)
- **Over-the-cap path**: Re-sign own FAs via Bird rights, use $15M non-taxpayer MLE

The path chosen determines the apron status, which constrains all trade activity.


In [2]:
# Tradeable contracts
print("=== Lakers Tradeable Contracts (2026-27) ===")
tradeable_df = pd.DataFrame([
    {"Player": k, "Salary": v, "Salary_Fmt": f"${v:,.0f}"}
    for k, v in LAKERS_TRADEABLE.items()
])
tradeable_df = tradeable_df.sort_values('Salary', ascending=False)
print(tradeable_df[['Player', 'Salary_Fmt']].to_string(index=False))
print(f"\nTotal tradeable salary: ${tradeable_df['Salary'].sum():,.0f}")
print("\nUntradeable: Luka Doncic ($49.8M), Austin Reaves (FA until re-signed)")


=== Lakers Tradeable Contracts (2026-27) ===
           Player  Salary_Fmt
Jarred Vanderbilt $12,428,571
     Jake LaRavia  $6,000,000
    Dalton Knecht  $4,200,000
     Bronny James  $2,300,000
      Adou Thiero  $2,150,000

Total tradeable salary: $27,078,571

Untradeable: Luka Doncic ($49.8M), Austin Reaves (FA until re-signed)


In [3]:
# Cap scenarios
print("=== 2026-27 Cap Scenarios ===\n")
for name, scen in lakers_cap_scenarios().items():
    committed = scen.get('approx_committed', 0)
    cap_room = scen.get('cap_room', False)
    print(f"Scenario: {scen['description']}")
    print(f"  Apron status:    {scen['apron_status']}")
    print(f"  Approx committed: ${committed/1e6:.0f}M")
    print(f"  Cap room:        {'Yes' if cap_room else 'No'}")
    print(f"  Note:            {scen['note']}")
    print()


=== 2026-27 Cap Scenarios ===

Scenario: Renounce all FAs except Reaves
  Apron status:    below_cap
  Approx committed: $47M
  Cap room:        Yes
  Note:            ~$47M in cap room. Re-sign Reaves using Bird rights.

Scenario: Renounce ALL FAs including Reaves
  Apron status:    below_cap
  Approx committed: $67M
  Cap room:        Yes
  Note:            ~$67M in cap room. Reaves signs elsewhere.

Scenario: Re-sign own FAs using Bird rights, stay over cap
  Apron status:    over_cap
  Approx committed: $150M
  Cap room:        No
  Note:            No cap room. Use $15M non-taxpayer MLE to sign additional players.

Scenario: Re-sign FAs and add via MLE, land between aprons
  Apron status:    first_apron
  Approx committed: $210M
  Cap room:        No
  Note:            100% salary matching. Tax MLE only (~$6.1M). Aggregation still allowed.



In [4]:
# Extension-eligible players
print("=== Extension-Eligible Players ===")
ext_df = pd.DataFrame([
    {"Player": k, "Max Extension": v['max_extension'], "Deadline": v['deadline']}
    for k, v in LAKERS_EXTENSION_ELIGIBLE.items()
])
print(ext_df.to_string(index=False))
print("\nKey: If any of these are extended before a trade, their new salary changes the math.")


=== Extension-Eligible Players ===
           Player Max Extension         Deadline
    Austin Reaves    4yr/$87.4M          June 30
    Rui Hachimura   4yr/$114.5M          June 30
      Maxi Kleber      4yr/$87M          June 30
Jarred Vanderbilt    4yr/$92.8M          Sept 18
     Bronny James    4yr/$92.8M Day after Finals

Key: If any of these are extended before a trade, their new salary changes the math.


## C. CBA Salary Matching Rules

The NBA's 2023 CBA governs what trades are legal. Three salary tiers apply depending on
the team's total payroll relative to the apron thresholds.

**Key insight**: The Lakers will almost certainly be above the first apron (between $199M-$212M
in total payroll once all FAs are re-signed). This means:
- **100% matching rule**: Outgoing salary must be >= incoming salary
- **Aggregation allowed**: Can combine multiple players to match one incoming player
- **No Traded Player Exceptions** (TPEs) can be used

If they exceed $212M (second apron), aggregation is also banned.


In [5]:
thresholds = {
    "Salary Cap": SALARY_CAP_2026_27,
    "First Apron": FIRST_APRON_2026_27,
    "Second Apron": SECOND_APRON_2026_27,
}
print("=== 2026-27 CBA Thresholds ===")
for k, v in thresholds.items():
    print(f"  {k}: ${v/1e6:.1f}M")


=== 2026-27 CBA Thresholds ===
  Salary Cap: $165.0M
  First Apron: $209.0M
  Second Apron: $222.0M


In [6]:
# Worked examples of CBA matching at each tier
examples = [
    {"desc": "Below both aprons: send $5M, receive $9.5M",   "out": 5_000_000,  "in": 9_500_000,  "status": "below_aprons", "n": 1},
    {"desc": "Below both aprons: send $5M, receive $10.5M",  "out": 5_000_000,  "in": 10_500_000, "status": "below_aprons", "n": 1},
    {"desc": "First apron: send $12.4M, receive $15M",       "out": 12_428_571, "in": 15_000_000, "status": "first_apron",  "n": 1},
    {"desc": "First apron: send $16.6M (2 players), rcv $15M","out": 16_628_571, "in": 15_000_000, "status": "first_apron", "n": 2},
    {"desc": "Second apron: send $16.6M (2 players), rcv $15M","out": 16_628_571,"in": 15_000_000,"status": "second_apron","n": 2},
    {"desc": "Second apron: send $12.4M (1 player), rcv $12M","out": 12_428_571, "in": 12_000_000, "status": "second_apron","n": 1},
]

print("=== CBA Matching Rule Examples ===\n")
for ex in examples:
    ok, reason, _ = trade_is_legal(ex['out'], ex['in'], 200_000_000, ex['status'], ex['n'])
    verdict = 'LEGAL' if ok else 'ILLEGAL'
    print(f"[{verdict}] {ex['desc']}")
    print(f"         {reason[:120]}")
    print()


=== CBA Matching Rule Examples ===

[LEGAL] Below both aprons: send $5M, receive $9.5M
         Legal. Outgoing $5,000,000 permits up to $10,250,000 incoming; receiving $9,500,000.

[ILLEGAL] Below both aprons: send $5M, receive $10.5M
         Salary match exceeded. Outgoing: $5,000,000 allows max incoming of $10,250,000, but incoming is $10,500,000.

[ILLEGAL] First apron: send $12.4M, receive $15M
         100% matching rule violated. Outgoing: $12,428,571, Incoming: $15,000,000. Above first/second apron, outgoing must be >=

[LEGAL] First apron: send $16.6M (2 players), rcv $15M
         Legal. Outgoing $16,628,571 >= Incoming $15,000,000.

[ILLEGAL] Second apron: send $16.6M (2 players), rcv $15M
         Aggregation banned above second apron. Cannot combine multiple salaries to match one incoming player.

[LEGAL] Second apron: send $12.4M (1 player), rcv $12M
         Legal. Outgoing $12,428,571 >= Incoming $12,000,000. Note: Cannot include cash. Cannot use sign-and-trade. Cannot

## D. Draft Pick Valuation

Draft picks are the Lakers' primary trade currency beyond their limited contracts.
Their pick situation is unusual: **the distant picks (2031, 2033) project as MORE
valuable to trade partners than a typical distant pick would be.**

**Why**: The Lakers will be good now (Luka at 26-28), so near-term picks project
as late firsts (pick 20-30, worth ~$1.5-4.5M surplus). But by 2031-33, Luka is 32-34
and the team may be declining or rebuilding -- those picks project as mid-lottery.

This mirrors the **Nets-Celtics dynamic** from 2013: Brooklyn's distant picks became
Jaylen Brown (#3) and Jayson Tatum (#3) because the Nets aged out of contention.

We use the expected draft position range for each specific Lakers pick rather than
applying a generic time discount.


In [7]:
# Display the pick value curve
curve_df = pd.DataFrame([
    {"Pick": k, "Surplus Value ($M)": v}
    for k, v in PICK_VALUE_CURVE.items()
])

fig = px.line(
    curve_df, x="Pick", y="Surplus Value ($M)",
    title="Draft Pick Value Curve (surplus over 4-year rookie deal)",
    color_discrete_sequence=["#FDB927"],
    template="plotly_dark",
)
fig.update_layout(
    xaxis_title="Pick Number",
    yaxis_title="Estimated Surplus Value ($M)",
    plot_bgcolor="#1a1a2e",
    paper_bgcolor="#1a1a2e",
)
fig.show()


In [8]:
# Lakers specific pick values
print("=== Lakers Tradeable Pick Values ===\n")
for row in lakers_tradeable_picks_summary():
    rng = row['expected_range']
    rng_str = f"#{rng[0]}-{rng[1]}" if rng[0] != rng[1] else f"#{rng[0]}"
    print(f"  {row['year']} Round {row['round']} ({rng_str}): ${row['estimated_value_M']:.1f}M")
    print(f"    {row['note']}")
    print()

print("NOT tradeable:")
for key, info in LAKERS_DRAFT_PICKS.items():
    if not info.get('tradeable'):
        print(f"  {info['year']} 1st: {info['note']}")


=== Lakers Tradeable Pick Values ===

  2026 Round 1 (#25): $2.5M
    Known position - most liquid asset.

  2031 Round 1 (#8-20): $7.7M
    Luka will be 32. Window likely closing. Projects as mid-lottery.

  2032 Round 1 (#5-18): $9.8M
    Cannot trade in combination with 2031 or 2033 (Stepien Rule). Luka 33.

  2032 Round 2 (#31-60): $0.5M
    Second-round pick.

  2033 Round 1 (#3-16): $11.9M
    Luka 34. Possible rebuild territory. Most valuable distant pick.

NOT tradeable:
  2027 1st: Top-4 protected to Memphis. Lakers keep only if pick lands 1-4.
  2029 1st: Unprotected to Dallas. Not available to trade.


In [9]:
# Stepien Rule check for meaningful pick combinations
print("=== Stepien Rule Compliance ===")
combos_to_check = [
    (['2026_1st_25'],              "2026 #25 alone"),
    (['2031_1st'],                 "2031 1st alone"),
    (['2033_1st'],                 "2033 1st alone"),
    (['2031_1st', '2033_1st'],    "2031 + 2033 together"),
    (['2031_1st', '2032_1st'],    "2031 + 2032 (violation)"),
    (['2032_1st', '2033_1st'],    "2032 + 2033 (violation)"),
]
for keys, desc in combos_to_check:
    ok, reason = check_stepien_rule(keys)
    total_val = pick_combination_value(keys)
    verdict = "OK" if ok else "VIOLATION"
    print(f"[{verdict}] {desc}: ${total_val:.1f}M | {reason}")


=== Stepien Rule Compliance ===
[OK] 2026 #25 alone: $2.5M | Pick combination is Stepien Rule compliant.
[OK] 2031 1st alone: $7.7M | Pick combination is Stepien Rule compliant.
[OK] 2033 1st alone: $11.9M | Pick combination is Stepien Rule compliant.
[OK] 2031 + 2033 together: $19.6M | Pick combination is Stepien Rule compliant.
[VIOLATION] 2031 + 2032 (violation): $17.5M | Stepien Rule violation: cannot trade '2032_1st' and '2031_1st' in the same deal (consecutive first-round picks).
[VIOLATION] 2032 + 2033 (violation): $21.6M | Stepien Rule violation: cannot trade '2032_1st' and '2033_1st' in the same deal (consecutive first-round picks).


## E. Sample Trade Scenarios

### E.1 Rim Runner Targets

The Lakers' most pressing need (per Phase 1 analysis) is a rim runner -- a lob threat
who can play next to Luka without needing the ball. Top candidates:
- **Daniel Gafford** (WAS): best archetype fit, but under contract at $17M
- **Goga Bitadze** (ORL): cheaper, young, good fit
- **Ryan Kalkbrenner** (CHA): cheapest option, rookie contract


In [10]:
RIM_RUNNER_TARGETS = ["Daniel Gafford", "Goga Bitadze", "Ryan Kalkbrenner"]

for target in RIM_RUNNER_TARGETS:
    row = df[df['PLAYER_NAME'] == target]
    if row.empty:
        print(f"{target}: not found in dataset")
        continue
    r = row.iloc[0]
    print(f"=== {target} ===")
    print(f"  Team: {r.get('TEAM_ABBREVIATION', 'N/A')} | Age: {r.get('AGE', 'N/A')}")
    sal = r.get('SALARY_FORWARD')
    print(f"  2026-27 Salary: {'${:,.0f}'.format(sal) if pd.notna(sal) else 'FA / unknown'}")
    print(f"  ON_COURT_VALUE: {r.get('ON_COURT_VALUE', float('nan')):.3f}")
    print(f"  SURPLUS_VALUE: {r.get('SURPLUS_VALUE', float('nan')):.3f}")
    print(f"  Archetype: {r.get('BEST_ARCHETYPE', 'N/A')} | Dist: {r.get('ARCHETYPE_DISTANCE', float('nan')):.2f}")
    is_rfa = target in RESTRICTED_FREE_AGENTS
    print(f"  RFA: {is_rfa}")
    print()


=== Daniel Gafford ===
  Team: DAL | Age: 27.0
  2026-27 Salary: $17,263,584
  ON_COURT_VALUE: 0.467
  SURPLUS_VALUE: 0.023
  Archetype: rim_runner | Dist: 2.14
  RFA: False

=== Goga Bitadze ===
  Team: ORL | Age: 26.0
  2026-27 Salary: $7,608,696
  ON_COURT_VALUE: 1.284
  SURPLUS_VALUE: 0.551
  Archetype: rim_runner | Dist: 1.36
  RFA: False

=== Ryan Kalkbrenner ===
  Team: CHA | Age: 24.0
  2026-27 Salary: $2,411,090
  ON_COURT_VALUE: 1.276
  SURPLUS_VALUE: 0.863
  Archetype: rim_runner | Dist: 1.63
  RFA: False



In [11]:
# Trade packages for Gafford under first apron rules
print("=== Packages for Daniel Gafford (first_apron) ===")
pkgs = build_trade_packages("Daniel Gafford", df, apron_status='first_apron', max_packages=8)
if pkgs:
    print(packages_to_dataframe(pkgs)[
        ['Outgoing Players', 'Outgoing Salary', 'Target Salary', 'Legal', 'Lakers Net Value', 'Feasibility']
    ].to_string(index=False))
else:
    print("No legal packages found.")


=== Packages for Daniel Gafford (first_apron) ===
                                               Outgoing Players Outgoing Salary Target Salary Legal Lakers Net Value Feasibility
                               Jarred Vanderbilt + Jake LaRavia     $18,428,571   $17,263,584   Yes           +0.979        0.00
                 Jarred Vanderbilt + Jake LaRavia + Adou Thiero     $20,578,571   $17,263,584   Yes           +0.979        0.00
                Jarred Vanderbilt + Jake LaRavia + Bronny James     $20,728,571   $17,263,584   Yes           +0.979        0.00
               Jarred Vanderbilt + Jake LaRavia + Dalton Knecht     $22,628,571   $17,263,584   Yes           +0.979        0.00
  Jarred Vanderbilt + Jake LaRavia + Bronny James + Adou Thiero     $22,878,571   $17,263,584   Yes           +0.979        0.00
 Jarred Vanderbilt + Jake LaRavia + Dalton Knecht + Adou Thiero     $24,778,571   $17,263,584   Yes           +0.979        0.00
Jarred Vanderbilt + Jake LaRavia + Dalton Knech

In [12]:
# With a 2031 first included
print("=== Packages for Daniel Gafford + 2031 1st pick ===")
pkgs_w_pick = build_trade_packages(
    "Daniel Gafford", df,
    apron_status='first_apron',
    include_picks=['2031_1st'],
    max_packages=5,
)
if pkgs_w_pick:
    print(packages_to_dataframe(pkgs_w_pick)[
        ['Outgoing Players', 'Outgoing Salary', 'Picks', 'Pick Value ($M)', 'Feasibility']
    ].to_string(index=False))


=== Packages for Daniel Gafford + 2031 1st pick ===
                                              Outgoing Players Outgoing Salary    Picks Pick Value ($M) Feasibility
               Jarred Vanderbilt + Dalton Knecht + Adou Thiero     $18,778,571 2031_1st             7.7        0.03
              Jarred Vanderbilt + Dalton Knecht + Bronny James     $18,928,571 2031_1st             7.7        0.03
Jarred Vanderbilt + Dalton Knecht + Bronny James + Adou Thiero     $21,078,571 2031_1st             7.7        0.03
                              Jarred Vanderbilt + Jake LaRavia     $18,428,571 2031_1st             7.7        0.00
                Jarred Vanderbilt + Jake LaRavia + Adou Thiero     $20,578,571 2031_1st             7.7        0.00


In [13]:
# Small trade: Ryan Kalkbrenner ($2.4M, CHA)
print("=== Packages for Ryan Kalkbrenner (first_apron) ===")
pkgs_k = build_trade_packages("Ryan Kalkbrenner", df, apron_status='first_apron', max_packages=5)
if pkgs_k:
    print(packages_to_dataframe(pkgs_k)[
        ['Outgoing Players', 'Outgoing Salary', 'Legal', 'Feasibility']
    ].to_string(index=False))
else:
    # Fallback: show salary match options
    row = df[df['PLAYER_NAME'] == 'Ryan Kalkbrenner']
    if not row.empty:
        sal = row.iloc[0].get('SALARY_FORWARD', 2_400_000)
        if pd.isna(sal):
            sal = 2_400_000
        pkgs_raw = salary_match_options(float(sal), apron_status='first_apron')
        print(f"Salary match options for ${sal:,.0f}:")
        for p in pkgs_raw[:5]:
            print(' ', p)


=== Packages for Ryan Kalkbrenner (first_apron) ===
                                             Outgoing Players Outgoing Salary Legal Feasibility
                             Jarred Vanderbilt + Jake LaRavia     $18,428,571   Yes        0.00
               Jarred Vanderbilt + Jake LaRavia + Adou Thiero     $20,578,571   Yes        0.00
              Jarred Vanderbilt + Jake LaRavia + Bronny James     $20,728,571   Yes        0.00
             Jarred Vanderbilt + Jake LaRavia + Dalton Knecht     $22,628,571   Yes        0.00
Jarred Vanderbilt + Jake LaRavia + Bronny James + Adou Thiero     $22,878,571   Yes        0.00


### E.2 Three-and-D Wing Targets

The Lakers also need wings who can guard multiple positions and shoot. Key targets:
- **Cam Spencer** (BKN): young shooter, under contract
- **Dalton Knecht**-type players from other teams


In [14]:
WING_TARGETS = ["Cam Spencer", "Caleb Martin", "Jalen Suggs"]

for target in WING_TARGETS:
    row = df[df['PLAYER_NAME'] == target]
    if row.empty:
        print(f"{target}: not found")
        continue
    r = row.iloc[0]
    sal = r.get('SALARY_FORWARD')
    print(f"{target} | {r.get('TEAM_ABBREVIATION')} | ${sal:,.0f}" if pd.notna(sal) else f"{target} | FA")
    print(f"  Value: {r.get('ON_COURT_VALUE', float('nan')):.3f} | Archetype: {r.get('BEST_ARCHETYPE', 'N/A')}")


Cam Spencer | MEM | $2,411,090
  Value: 0.601 | Archetype: secondary_creator
Caleb Martin | DAL | $10,001,493
  Value: nan | Archetype: nan
Jalen Suggs | ORL | $32,400,000
  Value: 0.391 | Archetype: secondary_creator


In [15]:
# Show auto-generated packages for a mid-salary wing
TARGET_WING = "Jalen Suggs"
print(f"=== Packages for {TARGET_WING} (first_apron) ===")
pkgs_w = build_trade_packages(TARGET_WING, df, apron_status='first_apron', max_packages=6)
if pkgs_w:
    print(packages_to_dataframe(pkgs_w)[
        ['Outgoing Players', 'Outgoing Salary', 'Target Salary', 'Legal', 'Feasibility']
    ].to_string(index=False))
else:
    print("No legal packages found.")


=== Packages for Jalen Suggs (first_apron) ===
No legal packages found.


## F. Free Agent Scenarios

If the Lakers operate as a cap-space team, some targets are free agents -- no trade needed.
Others are Restricted Free Agents (RFAs) where the current team can match any offer sheet.


In [16]:
# Key FA targets -- no trade mechanics needed, just cap room
FA_TARGETS = ["Matisse Thybulle", "Bones Hyland", "Tobias Harris", "Kevon Looney"]

print("=== Cap-Space Free Agent Targets ===")
for name in FA_TARGETS:
    row = df[df['PLAYER_NAME'] == name]
    if row.empty:
        print(f"{name}: not in dataset")
        continue
    r = row.iloc[0]
    is_fa = bool(r.get('IS_FREE_AGENT', False))
    print(f"{name}: IS_FREE_AGENT={is_fa} | Value={r.get('ON_COURT_VALUE', float('nan')):.3f} | Arch={r.get('BEST_ARCHETYPE','N/A')}")


=== Cap-Space Free Agent Targets ===
Matisse Thybulle: IS_FREE_AGENT=True | Value=1.045 | Arch=three_and_d
Bones Hyland: IS_FREE_AGENT=True | Value=0.586 | Arch=secondary_creator
Tobias Harris: IS_FREE_AGENT=True | Value=0.689 | Arch=three_and_d
Kevon Looney: IS_FREE_AGENT=False | Value=nan | Arch=nan


In [17]:
# RFA targets -- offer sheet risk
print("=== Restricted Free Agent Targets (offer sheet risk) ===")
for name in RESTRICTED_FREE_AGENTS:
    row = df[df['PLAYER_NAME'] == name]
    if row.empty:
        print(f"{name}: not in dataset -- known RFA per ESPN")
        continue
    r = row.iloc[0]
    is_fa = bool(r.get('IS_FREE_AGENT', False))
    print(f"{name} | Current team can MATCH any offer sheet")
    print(f"  IS_FREE_AGENT={is_fa} | Value={r.get('ON_COURT_VALUE', float('nan')):.3f}")
    print(f"  Archetype: {r.get('BEST_ARCHETYPE', 'N/A')}")
    print()


=== Restricted Free Agent Targets (offer sheet risk) ===
Jalen Duren | Current team can MATCH any offer sheet
  IS_FREE_AGENT=True | Value=2.022
  Archetype: nan

Walker Kessler | Current team can MATCH any offer sheet
  IS_FREE_AGENT=True | Value=nan
  Archetype: nan

Peyton Watson | Current team can MATCH any offer sheet
  IS_FREE_AGENT=True | Value=0.099
  Archetype: three_and_d

Tari Eason | Current team can MATCH any offer sheet
  IS_FREE_AGENT=True | Value=-0.201
  Archetype: three_and_d



## G. "What Would It Take?" -- Trade vs. Free Agency Path

The Lakers face a build-vs-buy decision on each target archetype:
- **FA path** (cap space): sign players directly; no assets given up; requires renouncing Bird rights
- **Trade path**: give up contracts + picks; keep cap flexibility; can be done without cap room

Below we compare the "cost" of each path for a representative target per archetype.


In [18]:
# Top targets per archetype by combined rank
archetypes = ['rim_runner', 'three_and_d', 'shooter', 'secondary_creator']
non_lakers = df[df['TEAM_ABBREVIATION'] != 'LAL'].copy()

print("=== Top Target Per Archetype ===\n")
for arch in archetypes:
    subset = non_lakers[
        (non_lakers['BEST_ARCHETYPE'] == arch) &
        non_lakers['ON_COURT_VALUE'].notna()
    ].sort_values('AGE_ADJUSTED_SURPLUS', ascending=False)

    if subset.empty:
        continue
    r = subset.iloc[0]
    is_fa = bool(r.get('IS_FREE_AGENT', False))
    sal = r.get('SALARY_FORWARD')
    print(f"[{arch.replace('_', ' ').title()}] {r['PLAYER_NAME']} ({r.get('TEAM_ABBREVIATION', '?')})")
    print(f"  Age: {r.get('AGE', '?')} | Salary: {'${:,.0f}'.format(sal) if pd.notna(sal) else 'FA'}")
    print(f"  Value: {r.get('ON_COURT_VALUE', float('nan')):.3f} | Surplus: {r.get('SURPLUS_VALUE', float('nan')):.3f}")
    print(f"  FA path available: {is_fa}")
    if not is_fa and pd.notna(sal):
        pkgs = build_trade_packages(r['PLAYER_NAME'], df, apron_status='first_apron', max_packages=1)
        if pkgs:
            best = pkgs[0]
            print(f"  Best trade package: {' + '.join(best.outgoing_players)} (feasibility: {best.feasibility_score:.2f})")
        else:
            print(f"  Trade path: no salary-matching package found at first_apron")
    print()


=== Top Target Per Archetype ===

[Rim Runner] Ryan Kalkbrenner (CHA)
  Age: 24.0 | Salary: $2,411,090
  Value: 1.276 | Surplus: 0.863
  FA path available: False
  Best trade package: Jarred Vanderbilt + Jake LaRavia (feasibility: 0.00)

[Three And D] Jordan Walsh (BOS)
  Age: 22.0 | Salary: $2,406,205
  Value: 0.578 | Surplus: 0.679
  FA path available: False
  Best trade package: Jarred Vanderbilt + Jake LaRavia (feasibility: 0.00)

[Shooter] Simone Fontecchio (MIA)
  Age: 30.0 | Salary: FA
  Value: 0.233 | Surplus: 0.617
  FA path available: True

[Secondary Creator] Ajay Mitchell (OKC)
  Age: 23.0 | Salary: $2,850,000
  Value: 1.027 | Surplus: 0.749
  FA path available: False
  Best trade package: Jarred Vanderbilt + Jake LaRavia (feasibility: 0.00)



## H. Methodology Notes and Limitations

### CBA Rules
- Rules reflect the **2026-27 CBA** with projected thresholds (ESPN Bobby Marks, May 2026).
- First apron ($199M) / Second apron ($212M) figures are projected; actual NBA announcement expected July 2026.
- The model encodes rules as functions parameterized by apron status; when official figures are released, update `constants.py` only.

### Pick Valuation
- Pick values use a **static surplus curve** calibrated to historical NBA data (Pelton/Hollinger-style).
- Lakers 2031/2033 firsts are modeled using their **expected draft position range** (mid-lottery), not a generic time discount.
- Expected ranges reflect the assumption that the Luka window closes by 2031-33 -- an assumption that could be wrong.
- A Monte Carlo protection model (for the 2027 Memphis pick) is aspirational for a future phase.

### Trade Feasibility Score
- Feasibility is a simple heuristic: compare value received by the other team to the target's on-court value.
- Real trades involve organizational priorities, chemistry considerations, media markets, and negotiation dynamics not modeled here.
- The 110-130% premium rule is a useful first-pass filter, not a prediction.

### Free Agency
- Model flags IS_FREE_AGENT based on whether a 2026-27 salary record exists in the BBRef CSV.
- Some players' FA status may be incorrect if they were traded or have unexercised options not yet reflected.
- RFA status is manually encoded from ESPN (Duren, Kessler, Watson, Eason).

### What's Next (Phase 3)
- Improve pick valuation with Monte Carlo simulation for protected picks.
- Add play-type data (roll-man, transition) for more precise archetype matching.
- Model multi-team trades.
- Dashboard polish (Phase 3).
